# 02 — Extrair 1.000 patentes para o treinamento LoRA

Execute no **EPO Technology Intelligence Platform — PATSTAT**. Este notebook não baixa a rede de citações; ele busca somente os dados necessários ao ajuste LoRA: título, resumo e rótulo CPC.

São sorteadas inicialmente 3.500 famílias candidatas porque algumas serão excluídas por ausência de texto, CPC ou por terem a primeira aplicação fora de 2010–2024. A saída final contém exatamente 1.000 patentes distintas, divididas em 800/100/100.

In [ ]:
from epo.tipdata.patstat import PatstatClient
from sqlalchemy import text
from pathlib import Path
import json
import shutil
import pandas as pd

patstat = PatstatClient(env='PROD')
db = patstat.orm()

def consultar(sql):
    resultado = db.execute(text(sql))
    return pd.DataFrame(resultado.fetchall(), columns=resultado.keys())

def blocos(valores, tamanho=350):
    valores = list(dict.fromkeys(int(v) for v in valores))
    for inicio in range(0, len(valores), tamanho):
        yield valores[inicio:inicio + tamanho]

def consultar_em_blocos(modelo_sql, valores, tamanho=350):
    partes = []
    for numero, bloco in enumerate(blocos(valores, tamanho), start=1):
        ids = ','.join(map(str, bloco))
        partes.append(consultar(modelo_sql.format(ids=ids)))
        print('Bloco', numero, 'concluído')
    return pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()

print('Conexão concluída.')

In [ ]:
AREA = 'ambas'  # biotecnologia, saude_farmaceutica ou ambas
ANO_INICIAL = 2010
ANO_FINAL = 2024
ALVO_LORA = 1000
AMOSTRA_CANDIDATA = 3500
SEMENTE = 42

filtros_area = {
    'biotecnologia': "(i.ipc_class_symbol LIKE 'C12N%' OR i.ipc_class_symbol LIKE 'C12Q%')",
    'saude_farmaceutica': "i.ipc_class_symbol LIKE 'A61%'",
    'ambas': "(i.ipc_class_symbol LIKE 'A61%' OR i.ipc_class_symbol LIKE 'C12N%' OR i.ipc_class_symbol LIKE 'C12Q%')",
}
filtro_area = filtros_area[AREA]
criterio_brasil = """
(
  a.appln_auth = 'BR'
  OR EXISTS (
    SELECT 1 FROM tls207_pers_appln pa
    JOIN tls206_person p ON p.person_id = pa.person_id
    WHERE pa.appln_id = a.appln_id
      AND p.person_ctry_code = 'BR'
      AND (pa.applt_seq_nr > 0 OR pa.invt_seq_nr > 0)
  )
)
"""
print('Meta:', ALVO_LORA, '| candidatos iniciais:', AMOSTRA_CANDIDATA)

In [ ]:
sql_familias = f"""
SELECT DISTINCT a.docdb_family_id
FROM tls201_appln a
JOIN tls209_appln_ipc i ON i.appln_id = a.appln_id
WHERE a.docdb_family_id > 0
  AND a.appln_filing_year BETWEEN {ANO_INICIAL} AND {ANO_FINAL}
  AND {criterio_brasil}
  AND {filtro_area}
"""
disponiveis = consultar(sql_familias)['docdb_family_id'].dropna().astype('int64').drop_duplicates()
if len(disponiveis) < ALVO_LORA:
    raise ValueError(f'O universo encontrado tem somente {len(disponiveis)} famílias.')
n = min(AMOSTRA_CANDIDATA, len(disponiveis))
familias = disponiveis.sample(n=n, random_state=SEMENTE).sort_values().tolist()
print('Famílias disponíveis:', len(disponiveis))
print('Candidatas sorteadas:', len(familias))

In [ ]:
sql_aplicacoes = """
SELECT a.docdb_family_id AS patent_id, a.appln_id, a.appln_auth,
       a.appln_nr, a.appln_filing_date AS application_date,
       a.appln_filing_year AS application_year
FROM tls201_appln a
WHERE a.docdb_family_id IN ({ids})
"""
sql_titulos = """
SELECT a.docdb_family_id AS patent_id, t.appln_title AS title,
       t.appln_title_lg AS language
FROM tls201_appln a JOIN tls202_appln_title t ON t.appln_id = a.appln_id
WHERE a.docdb_family_id IN ({ids})
"""
sql_resumos = """
SELECT a.docdb_family_id AS patent_id, r.appln_abstract AS abstract,
       r.appln_abstract_lg AS language
FROM tls201_appln a JOIN tls203_appln_abstr r ON r.appln_id = a.appln_id
WHERE a.docdb_family_id IN ({ids})
"""
sql_cpc = """
SELECT DISTINCT a.docdb_family_id AS patent_id, c.cpc_class_symbol
FROM tls201_appln a JOIN tls224_appln_cpc c ON c.appln_id = a.appln_id
WHERE a.docdb_family_id IN ({ids})
"""

print('Consultando aplicações...')
aplicacoes = consultar_em_blocos(sql_aplicacoes, familias)
print('Consultando títulos...')
titulos_brutos = consultar_em_blocos(sql_titulos, familias)
print('Consultando resumos...')
resumos_brutos = consultar_em_blocos(sql_resumos, familias)
print('Consultando CPC...')
cpcs = consultar_em_blocos(sql_cpc, familias)

In [ ]:
aplicacoes['application_date'] = pd.to_datetime(aplicacoes['application_date'], errors='coerce')
aplicacoes['application_year'] = pd.to_numeric(aplicacoes['application_year'], errors='coerce')
representantes = (
    aplicacoes.sort_values(['patent_id', 'application_date', 'appln_id'])
    .drop_duplicates('patent_id', keep='first')
)
patentes = representantes[[
    'patent_id', 'appln_id', 'appln_auth', 'appln_nr',
    'application_date', 'application_year'
]].copy()
patentes = patentes[patentes['application_year'].between(ANO_INICIAL, ANO_FINAL)]

def escolher_texto(df, coluna):
    x = df.dropna(subset=[coluna]).copy()
    x['ordem_idioma'] = x['language'].astype(str).str.upper().map({'PT': 1, 'EN': 2}).fillna(3)
    x['tamanho'] = x[coluna].astype(str).str.len()
    x = x.sort_values(
        ['patent_id', 'ordem_idioma', 'tamanho'], ascending=[True, True, False]
    ).drop_duplicates('patent_id')
    return x[['patent_id', coluna, 'language']].rename(
        columns={'language': f'{coluna}_language'}
    )

titulos = escolher_texto(titulos_brutos, 'title')
resumos = escolher_texto(resumos_brutos, 'abstract')
cpc_agregado = (
    cpcs.dropna(subset=['cpc_class_symbol'])
    .groupby('patent_id')['cpc_class_symbol']
    .agg(lambda s: sorted(set(map(str, s))))
    .reset_index(name='cpc_codes')
)
# Definição operacional reproduzível: primeiro símbolo CPC em ordem alfanumérica.
cpc_agregado['cpc_primary'] = cpc_agregado['cpc_codes'].str[0]

validas = (
    patentes.merge(titulos, on='patent_id', how='inner')
    .merge(resumos, on='patent_id', how='inner')
    .merge(cpc_agregado, on='patent_id', how='inner')
    .dropna(subset=['title', 'abstract', 'cpc_primary'])
    .drop_duplicates('patent_id')
)
validas = validas[
    validas['title'].astype(str).str.strip().ne('')
    & validas['abstract'].astype(str).str.strip().ne('')
]
print('Famílias candidatas válidas:', len(validas))
if len(validas) < ALVO_LORA:
    raise ValueError(
        f'Obtivemos {len(validas)} válidas. Aumente AMOSTRA_CANDIDATA e execute novamente.'
    )

In [ ]:
amostra = validas.sample(n=ALVO_LORA, random_state=SEMENTE).reset_index(drop=True)
amostra = amostra.sample(frac=1, random_state=SEMENTE).reset_index(drop=True)
treino = amostra.iloc[:800].copy()
validacao = amostra.iloc[800:900].copy()
teste = amostra.iloc[900:1000].copy()

def registro_lora(row):
    return {
        'instruction': 'Identifique a categoria CPC principal da patente.',
        'input': f"Titulo: {row['title']}\nResumo: {row['abstract']}",
        'output': str(row['cpc_primary']),
        'patent_id': str(row['patent_id']),
    }

saida = Path('LLM_EIDT_LORA_1000')
saida.mkdir(exist_ok=True)
amostra.to_parquet(saida / 'patents_lora_1000.parquet', index=False)
amostra.to_csv(saida / 'patents_lora_1000_preview.csv', index=False, encoding='utf-8-sig')

for nome, frame in [('train', treino), ('validation', validacao), ('test', teste)]:
    with (saida / f'lora_{nome}.jsonl').open('w', encoding='utf-8') as arquivo:
        for _, row in frame.iterrows():
            arquivo.write(json.dumps(registro_lora(row), ensure_ascii=False) + '\n')

manifesto = {
    'total': len(amostra), 'train': len(treino),
    'validation': len(validacao), 'test': len(teste),
    'seed': SEMENTE, 'area': AREA,
    'start_year': ANO_INICIAL, 'end_year': ANO_FINAL,
    'cpc_label_rule': 'primeiro símbolo CPC da família em ordem alfanumérica',
}
(saida / 'manifest.json').write_text(
    json.dumps(manifesto, ensure_ascii=False, indent=2), encoding='utf-8'
)
arquivo_zip = shutil.make_archive('LLM_EIDT_LORA_1000', 'zip', saida)

assert len(amostra) == 1000
assert amostra['patent_id'].nunique() == 1000
print('Treino:', len(treino), '| validação:', len(validacao), '| teste:', len(teste))
print('ZIP criado:', arquivo_zip)
print('Baixe o ZIP pelo painel de arquivos à esquerda.')